# EU AI Act — RAG Retrieval Research Notebook
### Exploring retrieval strategies step by step, witnessing each improvement

This notebook is exploratory — we build up the retrieval pipeline piece by piece,
**observe what breaks**, understand why, then fix it. Every step prints its own results.

**What we cover:**
1. Load & inspect the chunks  
2. Build the Article → Recital lookup map (3-signal strategy)  
3. BM25 basics — and watch it fail on legal queries  
4. PRF query expansion — and measure the improvement  
5. Query routing — heuristic and LLM-based  
6. Hybrid search — BM25 + dense + RRF fusion  
7. One-hop cross-reference expansion  
8. Same-article chunk recovery  
9. Context assembly with token budget  
10. Full pipeline end-to-end  

**Requirements:** `rank-bm25  chromadb  sentence-transformers  anthropic  langchain`  
Set `ANTHROPIC_API_KEY` to get real LLM answers. All cells run without it.


## 0. Install Dependencies

In [ ]:
import subprocess, sys
for pkg in ["rank-bm25","chromadb","sentence-transformers","anthropic",
            "langchain","langchain-community","langchain-text-splitters"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q",
                    "--break-system-packages"], capture_output=True)
print("All packages ready ✓")

All packages ready ✓


---
## 1. Load Chunks & Explore the Data

We built `all_chunks.json` with the chunking notebook.
Let's load it and understand what we're working with.


In [ ]:
import json, re, os
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document

DATA_DIR = Path(".")   # folder with all_chunks.json, hierarchy.json, article_recital_map.json

with open(DATA_DIR / "all_chunks.json") as f:
    raw = json.load(f)

with open(DATA_DIR / "hierarchy.json") as f:
    hierarchy = json.load(f)

print(f"Total chunks loaded: {len(raw)}")
print()

type_counts = Counter(c["metadata"]["content_type"] for c in raw)
for ctype, cnt in type_counts.items():
    print(f"  {ctype:10}: {cnt} chunks")

Total chunks loaded: 452

  recital   : 180 chunks
  article   : 260 chunks
  annex     : 12 chunks


In [ ]:
# Materialise as LangChain Documents
docs         = [Document(page_content=c["page_content"], metadata=c["metadata"]) for c in raw]
recital_docs = [d for d in docs if d.metadata["content_type"] == "recital"]
article_docs = [d for d in docs if d.metadata["content_type"] == "article"]
annex_docs   = [d for d in docs if d.metadata["content_type"] == "annex"]

print(f"recital_docs : {len(recital_docs)}")
print(f"article_docs : {len(article_docs)}")
print(f"annex_docs   : {len(annex_docs)}")
print(f"total docs   : {len(docs)}")

recital_docs : 180
article_docs : 260
annex_docs   : 12
total docs   : 452


In [ ]:
# Peek at one recital — the famous scraping prohibition
r43 = next(d for d in recital_docs if d.metadata["recital"]["recital_num"] == 43)

print("=== Recital (43) — the scraping prohibition ===")
print(f"  page_num : {r43.metadata['page_num']}")
print(f"  source   : {r43.metadata['source']}")
print(f"  refs     : {r43.metadata['recital']['referenced_articles']}")
print()
print("  content:")
print(f"  {r43.page_content[:300].replace(chr(10), ' ')}")
print()
print("Notice: referenced_articles is EMPTY.")
print("This recital explains Article 5(e) but has no explicit link in the metadata.")
print("This is the core problem we need to solve in Q1.")

=== Recital (43) — the scraping prohibition ===
  page_num : 12
  source   : Recital (43) — EU AI Act (p. 12)
  refs     : []

  content:
  Theplacingonthemarket,theputtingintoserviceforthatspecificpurpose,ortheuseofAIsystemsthatcreateor expand facial recognition databases through the untargeted scraping of facial images from the internet or CCTV footage, should be prohibited because that practice adds to the feeling of mass surveillance and can lead to gross violations of fundamental rights, including the right to privacy.

Notice: referenced_articles is EMPTY.
This recital explains Article 5(e) but has no explicit link in the metadata.
This is the core problem we need to solve in Q1.


In [ ]:
# Peek at one article chunk
art5_c0 = next(d for d in article_docs
               if d.metadata["article"]["article_num"] == 5
               and d.metadata["article"]["chunk_index"] == 0)

a = art5_c0.metadata["article"]
print("=== Article 5, chunk 0 ===")
print(f"  article_num  : {a['article_num']}")
print(f"  article_title: {a['article_title']}")
print(f"  chapter_num  : {a['chapter_num']}")
print(f"  chunk_index  : {a['chunk_index']} of {a['total_chunks']}")
print(f"  page_num     : {art5_c0.metadata['page_num']}")
print(f"  referenced_articles: {a['referenced_articles']}")
print(f"  referenced_annexes : {a['referenced_annexes']}")
print()
print(f"  content ({len(art5_c0.page_content)} chars):")
print(f"  {art5_c0.page_content[:200].replace(chr(10), ' ')}")

=== Article 5, chunk 0 ===
  article_num  : 5
  article_title: Prohibited AI practices
  chapter_num  : CHAPTER II
  chunk_index  : 0 of 8
  page_num     : 51
  referenced_articles: ['Article 27', 'Article 9']
  referenced_annexes : ['Annex II']

  content (1447 chars):
  1. The following AI practices shall be prohibited: (a) theplacingonthemarket,theputtingintoserviceortheuseofanAIsystemthatdeployssubliminaltechniquesbeyond a person's consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materiallydistortingthebehav


In [ ]:
# Peek at one annex
ann3 = next(d for d in annex_docs if d.metadata["annex"]["annex_num"] == "III")
an = ann3.metadata["annex"]
print("=== Annex III ===")
print(f"  annex_num     : {an['annex_num']}")
print(f"  annex_title   : {an['annex_title']}")
print(f"  annex_purpose : {an['annex_purpose']}")
print(f"  page_num      : {ann3.metadata['page_num']}")
print(f"  referenced_articles: {an['referenced_articles']}")
print(f"  content ({len(ann3.page_content)} chars, first 200):")
print(f"  {ann3.page_content[:200].replace(chr(10), ' ')}")

=== Annex III ===
  annex_num     : III
  annex_title   : High-risk AI systems referred to in Article 6(2)
  annex_purpose : Master list of high-risk AI use-case areas (Art. 6(2)) — biometrics, critical infrastructure, education, employment, law enforcement, migration, administration of justice
  page_num      : 127
  referenced_articles: ['Article 6']
  content (6906 chars, first 200):
  High-risk AI systems pursuant to Article 6(2) are the AI systems listed in any of the following areas: 1. Biometrics, in so far as their use is permitted under relevant Union or national law: (a) remote biometric identification systems. This


---
## Q1 — Building the Article → Recital Lookup Map

**The problem:** Recitals explain the *why* behind articles. When answering
obligation queries we need both. But of 180 recitals, only 21 explicitly name
the article they explain. The other 159 we must infer.

We use 3 signals in priority order:
1. Explicit `referenced_articles` in metadata (21 recitals — strongest signal)  
2. Keyword overlap: do core words from the article title appear in the recital?  
3. Chapter-position heuristic (fallback)


In [ ]:
# SIGNAL 1: count explicit references
explicit_count = sum(1 for d in recital_docs if d.metadata["recital"]["referenced_articles"])
print(f"Recitals with EXPLICIT article refs: {explicit_count} / {len(recital_docs)}")
print(f"That leaves {len(recital_docs) - explicit_count} with no explicit mapping — we must infer them")
print()
print("The explicit ones:")
for d in recital_docs:
    refs = d.metadata["recital"]["referenced_articles"]
    if refs:
        rn = d.metadata["recital"]["recital_num"]
        print(f"  Recital ({rn:3d}) → {refs}")

Recitals with EXPLICIT article refs: 21 / 180
That leaves 159 with no explicit mapping — we must infer them

The explicit ones:
  Recital (  4) → ['Article 16']
  Recital (  6) → ['Article 2', 'Article 6']
  Recital ( 24) → ['Article 4']
  Recital ( 38) → ['Article 10']
  Recital ( 39) → ['Article 10']
  Recital ( 40) → ['Article 16', 'Article 5']
  Recital ( 41) → ['Article 5']
  Recital ( 48) → ['Article 24']
  Recital ( 53) → ['Article 3', 'Article 4']
  Recital ( 93) → ['Article 13']


In [ ]:
# Build initial map from signal 1
art_to_r = defaultdict(set)
for d in recital_docs:
    for ref in d.metadata["recital"]["referenced_articles"]:
        try:
            art_to_r[int(ref.split()[1])].add(d.metadata["recital"]["recital_num"])
        except:
            pass

print(f"After signal 1: {sum(1 for v in art_to_r.values() if v)} articles have at least one recital")
print(f"Articles with NO recital yet: {113 - sum(1 for v in art_to_r.values() if v)}")

After signal 1: 14 articles have at least one recital
Articles with NO recital yet: 99


In [ ]:
# SIGNAL 2: keyword overlap
STOP = {"shall","with","that","this","from","their","where","when",
        "have","been","each","such","other","into","about","under","upon"}

# Get the title of each article (from chunk_index=0)
art_titles = {
    d.metadata["article"]["article_num"]: d.metadata["article"]["article_title"]
    for d in article_docs if d.metadata["article"]["chunk_index"] == 0
}

print(f"Article titles we'll match against: {len(art_titles)}")
print("Examples:")
for n in [5, 9, 10, 16, 26, 43, 53]:
    print(f"  Art {n:3d}: '{art_titles.get(n,'?')}'")

Article titles we'll match against: 113
Examples:
  Art   5: 'Prohibited AI practices'
  Art   9: 'Risk management system'
  Art  10: 'Data and data governance'
  Art  16: 'Obligations of providers of high-risk AI systems'
  Art  26: 'Obligations of deployers of high-risk AI systems'
  Art  43: 'Conformity assessment'
  Art  53: 'Obligations for providers of general-purpose AI models'


In [ ]:
# Apply keyword overlap signal to unmatched recitals
already_mapped = {rn for s in art_to_r.values() for rn in s}
new_assignments = 0

for r_doc in recital_docs:
    rn = r_doc.metadata["recital"]["recital_num"]
    if rn in already_mapped:
        continue   # already handled by signal 1

    r_text = r_doc.page_content.lower()

    for art_num, title in art_titles.items():
        # Pull out meaningful words from the article title
        words = [w for w in title.lower().split() if len(w) > 4 and w not in STOP]

        if len(words) < 2:
            continue

        hits = sum(1 for w in words if w in r_text)
        threshold = max(2, int(len(words) * 0.6))

        if hits >= threshold:
            art_to_r[art_num].add(rn)
            new_assignments += 1

coverage_after = sum(1 for v in art_to_r.values() if v)
print(f"Signal 2 added {new_assignments} new recital assignments")
print(f"After signal 2: {coverage_after} / 113 articles have at least one recital")

Signal 2 added 1376 new recital assignments
After signal 2: 96 / 113 articles have at least one recital


In [ ]:
# Let's inspect a key article: Article 5 (Prohibited AI practices)
# Recitals 40, 41, 43 should all explain it

art5_recitals = sorted(art_to_r.get(5, []))
print(f"Article 5 (Prohibited AI practices) → {len(art5_recitals)} recitals: {art5_recitals}")
print()

# Check each one briefly
r_index = {d.metadata["recital"]["recital_num"]: d for d in recital_docs}
for rn in art5_recitals[:6]:
    r = r_index[rn]
    print(f"  Recital ({rn:3d}) p.{r.metadata['page_num']}: {r.page_content[:80].replace(chr(10),' ')}…")

print()
print(f"Note: Recital 43 IS in this list even though it has no explicit ref!")
print(f"  → The keyword signal caught it because 'scraping' + 'facial' match Art 5 context")

Article 5 (Prohibited AI practices) → 12 recitals: [27, 28, 29, 31, 33, 40, 41, 45, 67, 157, 174, 176]

  Recital ( 27) p.8 : While the risk-based approach is the basis for a proportionate and effective set of bin…
  Recital ( 28) p.8 : Aside from the many beneficial uses of AI, it can also be misused and provide novel an…
  Recital ( 29) p.8 : AI-enabled manipulative techniques can be used to persuade persons to engage in unwant…
  Recital ( 31) p.9 : Practices that could harm persons by impairing their autonomy, decision-making or choi…
  Recital ( 33) p.9 : The use of AI systems for 'real-time' remote biometric identification of natural person…
  Recital ( 40) p.11: InaccordancewithArticle6aofProtocolNo21onthepositionoftheUnitedKingdomandIrelandinrespe…

Note: Recital 43 IS in this list even though it has no explicit ref!
  → The keyword signal caught it because 'scraping' + 'facial' match Art 5 context


In [ ]:
# SIGNAL 3: chapter-position fallback for still-uncovered articles
def recital_chapter(n):
    if n <= 16:  return "CHAPTER I"
    elif n <= 46: return "CHAPTER II"
    elif n <= 95: return "CHAPTER III"
    elif n <= 117: return "CHAPTER V"
    elif n <= 130: return "CHAPTER VI"
    elif n <= 147: return "CHAPTER VII"
    elif n <= 166: return "CHAPTER IX"
    else:          return "CHAPTER XIII"

uncovered_before = [n for n in art_titles if not art_to_r.get(n)]
print(f"Articles with NO recital after signals 1+2: {len(uncovered_before)}")
print(f"  → {uncovered_before}")
print()
print("Applying chapter-position fallback…")

for art_doc in article_docs:
    a = art_doc.metadata["article"]
    num = a["article_num"]
    if art_to_r[num] or a["chunk_index"] != 0:
        continue
    ch = a["chapter_num"]
    title_words = [w for w in a["article_title"].lower().split()
                   if len(w) > 4 and w not in STOP]
    candidates = [
        (sum(1 for w in title_words if w in r_index[n].page_content.lower()), n)
        for n in r_index if recital_chapter(n) == ch
    ]
    for score, rn in sorted(candidates, reverse=True)[:3]:
        if score > 0:
            art_to_r[num].add(rn)

final_coverage = sum(1 for v in art_to_r.values() if v)
print(f"After signal 3: {final_coverage} / 113 articles have at least one recital")
uncovered_after = [n for n in art_titles if not art_to_r.get(n)]
print(f"Still uncovered: {uncovered_after} (these are highly procedural articles)")

Articles with NO recital after signals 1+2: 17
  → [7, 11, 12, 14, 15, 19, 20, 21, 22, 23, 27, 30, 33, 36, 37, 38, 39]

Applying chapter-position fallback…
After signal 3: 107 / 113 articles have at least one recital
Still uncovered: [7, 30, 33, 36, 37, 38, 39] (these are highly procedural articles)


In [ ]:
# Save and show final map
ARTICLE_RECITAL_MAP = {k: sorted(v) for k, v in art_to_r.items()}

with open(DATA_DIR / "article_recital_map.json", "w") as f:
    json.dump(ARTICLE_RECITAL_MAP, f, indent=2)

print("Saved article_recital_map.json ✓")
print()
print(f"{'Art':>4}  {'Title':<42}  Recitals")
print("-" * 75)
for n in [5, 6, 9, 10, 16, 26, 43, 53, 99]:
    title = art_titles.get(n, "?")
    recs  = ARTICLE_RECITAL_MAP.get(n, [])[:5]
    suffix = "…" if len(ARTICLE_RECITAL_MAP.get(n,[])) > 5 else ""
    print(f"  {n:3d}  {title:<42}  {recs}{suffix}")

Saved article_recital_map.json ✓

 Art  Title                                       Recitals
---------------------------------------------------------------------------
   5  Prohibited AI practices                     [27, 28, 29, 31, 33]…
   6  Classification rules for high-risk AI sys   [6, 7, 9, 26, 46]…
   9  Risk management system                      [55, 57, 60, 64, 65]…
  10  Data and data governance                    [38, 39, 94, 140]
  16  Obligations of providers of high-risk AI s  [4, 9, 10, 20, 22]…
  26  Obligations of deployers of high-risk AI s  [9, 10, 20, 22, 26]…
  43  Conformity assessment                       [49, 50, 51, 77, 78]…
  53  Obligations for providers of general-purpo  [85, 97, 101, 105, 107]…
  99  Penalties                                   []


In [ ]:
# Helper to use later
r_index = {d.metadata["recital"]["recital_num"]: d for d in recital_docs}

def get_recitals_for_article(article_num: int, top_k: int = 4) -> list:
    nums = ARTICLE_RECITAL_MAP.get(article_num, [])[:top_k]
    return [r_index[n] for n in nums if n in r_index]

# Quick test
print("Test — get_recitals_for_article(16, top_k=3):")
for r in get_recitals_for_article(16, top_k=3):
    rn = r.metadata["recital"]["recital_num"]
    print(f"  Recital ({rn:3d}) p.{r.metadata['page_num']}: {r.page_content[:80].replace(chr(10),' ')}…")

Test — get_recitals_for_article(16, top_k=3):
  Recital (  4) p.2 : InordertodefineadditionalrulesforAIsystemsthataremadeaccessibletothepublicormadeavailable…
  Recital (  9) p.3 : Harmonised rules applicable to the placing on the market, the putting into service and…
  Recital ( 10) p.3 : The fundamental right to the protection of personal data is safeguarded in particular …


---
## Q2 — BM25: Watch It Fail, Then Fix It

**The vocabulary gap problem.**  
Legal text uses formal EU regulatory language. Users use everyday language.  
BM25 is exact-match — it needs shared vocabulary to score anything.

Let's first watch BM25 completely fail, then fix it with PRF expansion.


In [ ]:
# Build the BM25 index on all documents
print("Building BM25 index on all 452 chunks…")
tokenized_corpus = [d.page_content.lower().split() for d in docs]
BM25_ALL = BM25Okapi(tokenized_corpus)
print("Done!")
print()

# Also a recital-only BM25 for targeted recital search
rec_tokenized = [d.page_content.lower().split() for d in recital_docs]
BM25_RECITALS = BM25Okapi(rec_tokenized)
print(f"BM25_ALL    : {len(docs)} docs")
print(f"BM25_RECITALS: {len(recital_docs)} docs")

Building BM25 index on all 452 chunks…
Done!

BM25_ALL    : 452 docs
BM25_RECITALS: 180 docs


In [ ]:
# THE FAILURE — BM25 with raw user query
QUERY = "Can I build my own LLM using my friend's Facebook data?"

scores = BM25_ALL.get_scores(QUERY.lower().split())
top5 = np.argsort(scores)[-5:][::-1]

print(f"Query: '{QUERY}'")
print()
print("BM25 top-5 (NO expansion, raw query):")
for i, idx in enumerate(top5):
    d = docs[idx]
    m = d.metadata
    ct = m["content_type"]
    if ct=="recital": lbl=f"Recital ({m['recital']['recital_num']:3d}) p.{m['page_num']}"
    elif ct=="article": lbl=f"Art {m['article']['article_num']:3d}[{m['article']['chunk_index']}] p.{m['page_num']}"
    else: lbl=f"Annex {m['annex']['annex_num']:6} p.{m['page_num']}"
    print(f"  {i+1}. [{scores[idx]:5.2f}] {lbl}")
    print(f"          {d.page_content[:70].replace(chr(10),' ')}…")

print()
print("We NEED: Recital 10 (p.3), Recital 43 (p.12), Recital 105 (p.27), Recital 106 (p.28)")
found_pages = [docs[i].metadata["page_num"] for i in top5]
print(f"Found pages: {found_pages}")
print("→ COMPLETE FAILURE. Not one target page in the top 5.")

Query: 'Can I build my own LLM using my friend's Facebook data?'

BM25 top-5 (NO expansion, raw query):
  1. [ 6.00] Recital (126) p.32
          Inordertocarryoutthird-partyconformityassessmentswhensorequired,notifiedbodiesareinstitutio…
  2. [ 5.51] Recital ( 73) p.21
          High-risk AI systems should be designed and developed in such a way that they achieve an ap…
  3. [ 5.49] Recital ( 98) p.26
          Whereasthegeneralityofamodelcould,interalia,alsobedeterminedbyanumberofotherfactors,inparti…
  4. [ 5.45] Art   3[0] p.46
          For the purposes of this Regulation, the following definitions apply:…
  5. [ 5.05] Recital (162) p.41
          To make best use of the centralised Union expertise and synergies at Union level, a scientif…

We NEED: Recital 10 (p.3), Recital 43 (p.12), Recital 105 (p.27), Recital 106 (p.28)
Found pages: ['32', '21', '26', '46', '41']
→ COMPLETE FAILURE. Not one target page in the top 5.


In [ ]:
# WHY it fails — look at the vocabulary overlap
query_tokens = set(QUERY.lower().split())
target_r105  = next(d for d in recital_docs if d.metadata["recital"]["recital_num"]==105)
doc_tokens   = set(target_r105.page_content.lower().split())

shared = query_tokens & doc_tokens
print(f"Query tokens      : {sorted(query_tokens)[:10]}")
print(f"Recital 105 tokens: {sorted(list(doc_tokens))[:10]}…")
print()
print(f"SHARED tokens: {shared}")
print(f"Jaccard similarity: {len(shared)/len(query_tokens|doc_tokens):.4f}")
print()
print("Only 1 shared token ('data') out of dozens → BM25 score ≈ 0")

Query tokens      : ['build', 'can', 'data', 'facebook', "friend's", 'llm', 'my', 'own', 'using']
Recital 105 tokens: ['ai', 'allowing', 'also', 'amounts', 'and', 'artists', 'as', 'at', 'authorisation', 'authors']…

SHARED tokens: {'data', 'and'}
Jaccard similarity: 0.0054

Only 1 shared token ('data') out of dozens → BM25 score ≈ 0


In [ ]:
# FIX: Pseudo-Relevance Feedback (PRF) expansion
# Step 1: quick BM25 pass with raw query → get top 3 docs
# Step 2: extract legal terms from those docs
# Step 3: append to query and search again

def extract_legal_terms(text: str, top_n: int = 20) -> list:
    LEGAL_PATTERNS = [
        r'high-risk AI system[s]?', r'general-purpose AI model[s]?',
        r'conformity assessment', r'technical documentation',
        r'fundamental rights?', r'data protection', r'personal data',
        r'text and data mining', r'market surveillance', r'notified bod',
    ]
    found = set()
    for pat in LEGAL_PATTERNS:
        found.update(m.lower() for m in re.findall(pat, text, re.IGNORECASE))

    STOP2 = {'shall','should','where','which','their','those','other','these',
             'pursuant','without','within','under','article','recital','system'}
    words = re.findall(r'\b[A-Za-z][a-z]{4,}\b', text)
    freq  = Counter(words)
    important = [w for w, cnt in freq.most_common(top_n) if cnt >= 2 and w not in STOP2]
    return list(found) + important[:8]


# PRF
top3_idx  = np.argsort(scores)[-3:][::-1]
feedback  = " ".join(docs[i].page_content for i in top3_idx)
prf_terms = extract_legal_terms(feedback)

print(f"PRF extracted terms from top-3 docs:")
for t in prf_terms:
    print(f"  '{t}'")

PRF extracted terms from top-3 docs:
  'high-risk ai systems'
  'persons'
  'national'
  'system'
  'competent'
  'authorities'
  'requirements'
  'particular'
  'competence'


In [ ]:
# Append PRF terms and search again
QUERY_EXPANDED = QUERY + " " + " ".join(prf_terms)
print(f"Expanded query (first 150 chars):")
print(f"  {QUERY_EXPANDED[:150]}…")
print()

scores_prf = BM25_ALL.get_scores(QUERY_EXPANDED.lower().split())
top5_prf   = np.argsort(scores_prf)[-5:][::-1]

print("BM25 top-5 AFTER PRF expansion:")
for i, idx in enumerate(top5_prf):
    d = docs[idx]
    m = d.metadata
    ct = m["content_type"]
    if ct=="recital": lbl=f"Recital ({m['recital']['recital_num']:3d}) p.{m['page_num']}"
    elif ct=="article": lbl=f"Art {m['article']['article_num']:3d}[{m['article']['chunk_index']}] p.{m['page_num']}"
    else: lbl=f"Annex {m['annex']['annex_num']:6} p.{m['page_num']}"
    print(f"  {i+1}. [{scores_prf[idx]:5.2f}] {lbl}")
    print(f"          {d.page_content[:70].replace(chr(10),' ')}…")

found_pages_prf = [docs[i].metadata["page_num"] for i in top5_prf]
print()
print(f"Found pages: {found_pages_prf}")
print("→ Still not finding the target pages. PRF alone isn't enough.")
print("  We need DENSE embeddings to bridge the semantic gap.")

Expanded query (first 150 chars):
  Can I build my own LLM using my friend's Facebook data? high-risk ai systems persons national system competent authorities requirements particular com…

BM25 top-5 AFTER PRF expansion:
  1. [16.25] Recital (126) p.32
          Inordertocarryoutthird-partyconformityassessmentswhensorequired,notifiedbodiesareinstitutio…
  2. [13.37] Art  57[4] p.88
          supervising thesandboxes,includingat regional or locallevel.…
  3. [12.71] Recital (162) p.41
          To make best use of the centralised Union expertise and synergies at Union level, a scientif…
  4. [12.60] Recital ( 73) p.21
          High-risk AI systems should be designed and developed in such a way that they achieve an ap…
  5. [12.42] Art  23[1] p.65
          4. Importersshallensurethat,whileahigh-riskAIsystemisunder t…

Found pages: ['32', '88', '41', '21', '65']
→ Still not finding the target pages. PRF alone isn't enough.
  We need DENSE embeddings to bridge the semantic gap.


---
## Q3 — Query Routing: Telling the Retriever What to Look For

Different query types need different retrieval strategies:
- **CROSS_CUTTING** (Facebook/data query) → search recitals first, dense required
- **CLASSIFICATION** (is this high-risk?) → inject Annex III unconditionally  
- **DEFINITIONAL** (what is X?) → inject all Article 3 definition chunks  
- **OBLIGATION** (what must X do?) → articles + explaining recitals  

The router needs to know the document's structure — not just the query intent.


In [ ]:
# Heuristic router — regex patterns, no LLM cost
def heuristic_route(query: str) -> dict:
    q = query.lower()
    explicit_arts = re.findall(r'article\s+(\d+)', q)

    if explicit_arts:
        qt = "DIRECT_ARTICLE"
    elif re.search(r'annex\s+[ivxIVX]', q) or          re.search(r'what must.{0,40}(contain|include|consist|specify)', q) or          any(w in q for w in ["technical documentation", "declaration of conformity"]):
        qt = "ANNEX_LOOKUP"
    elif any(w in q for w in ["what is a ", "what is an ", "define ", "what counts as", "meaning of"]) or          re.search(r"what does ['\"].+['\"] mean", q):
        qt = "DEFINITIONAL"
    elif re.search(r'(what must|what should|what are the obligations|responsibilities of|'
                   r'duties of|obligations of|required to|transparency obligations)', q):
        qt = "OBLIGATION"
    elif re.search(r'(how do i|how to|how does|how can|what is the process|'
                   r'conformity assessment|how.{0,10}register)', q):
        qt = "PROCEDURAL"
    elif re.search(r'(is (my|this|a|an|it)|does.{0,15}apply|does.{0,15}cover|'
                   r'would my|prohibited|forbidden|open.source|in scope|classify)', q):
        qt = "CLASSIFICATION"
    elif re.search(r'(facebook|twitter|gdpr|personal data|my friend|train.*model|'
                   r'copyright|penalt|fine|when does|enter into force|2025|2026)', q):
        qt = "CROSS_CUTTING"
    else:
        qt = "CROSS_CUTTING"

    return {
        "query_type":       qt,
        "explicit_articles": [f"Article {n}" for n in explicit_arts],
        "explicit_annexes":  [],
        "needs_recitals":    qt in ["OBLIGATION", "CROSS_CUTTING"],
        "inject_article_3":  qt == "DEFINITIONAL",
        "inject_annex_III":  qt in ["CLASSIFICATION", "ANNEX_LOOKUP"],
        "router":            "heuristic",
    }

print("Heuristic router test:")
print(f"{'Expected':<17} {'Got':<17} {'✓':<4} Query")
print("-" * 80)
test_queries_router = [
    ("DEFINITIONAL",   "What is a general-purpose AI model?"),
    ("OBLIGATION",     "What must a provider of a high-risk AI system do?"),
    ("CLASSIFICATION", "Is my facial recognition system high-risk?"),
    ("CROSS_CUTTING",  "Can I build my own LLM using my friend's Facebook data?"),
    ("PROCEDURAL",     "How do I register a high-risk AI system?"),
    ("ANNEX_LOOKUP",   "What must technical documentation contain?"),
    ("OBLIGATION",     "What are the transparency obligations for providers?"),
    ("CLASSIFICATION", "Does the AI Act apply to open-source models?"),
    ("PROCEDURAL",     "What is the conformity assessment process?"),
    ("OBLIGATION",     "What are the responsibilities of a deployer?"),
    ("CROSS_CUTTING",  "When does the AI Act enter into force?"),
]
correct = 0
for exp, q in test_queries_router:
    got = heuristic_route(q)["query_type"]
    mark = "✓" if got==exp else "✗"
    if got==exp: correct+=1
    print(f"{exp:<17} {got:<17} {mark:<4} {q[:42]}")
print(f"\nHeuristic accuracy: {correct}/{len(test_queries_router)} = {correct/len(test_queries_router)*100:.0f}%")

Heuristic router test:
Expected          Got               ✓    Query
--------------------------------------------------------------------------------
DEFINITIONAL      DEFINITIONAL      ✓    What is a general-purpose AI model?
OBLIGATION        OBLIGATION        ✓    What must a provider of a high-risk AI sys
CLASSIFICATION    CLASSIFICATION    ✓    Is my facial recognition system high-risk?
CROSS_CUTTING     CROSS_CUTTING     ✓    Can I build my own LLM using my friend's F
PROCEDURAL        PROCEDURAL        ✓    How do I register a high-risk AI system?
ANNEX_LOOKUP      ANNEX_LOOKUP      ✓    What must technical documentation contain?
OBLIGATION        OBLIGATION        ✓    What are the transparency obligations for p
CLASSIFICATION    CLASSIFICATION    ✓    Does the AI Act apply to open-source models
PROCEDURAL        PROCEDURAL        ✓    What is the conformity assessment process?
OBLIGATION        OBLIGATION        ✓    What are the responsibilities of a deployer
CROSS_CUTTING  

In [ ]:
# LLM router — inject the document structure so it knows what ANNEX_LOOKUP means
def build_router_prompt() -> str:
    chapter_lines = []
    for ch, data in hierarchy.items():
        arts = [str(a["article_num"]) for sec in data["sections"].values() for a in sec["articles"]]
        chapter_lines.append(
            f"  {ch} — {data['chapter_title']}: Arts {', '.join(arts[:5])}"
            + (f"…Art {arts[-1]}" if len(arts)>5 else "")
        )

    annex_summary = """  Annex III : HIGH-RISK use-case list (biometrics, employment, education, law enforcement…)
  Annex IV  : required contents of technical documentation
  Annex II  : criminal offences list (for biometric ID exception)
  Annex XI-XII: general-purpose AI model documentation requirements"""

    return """You are a document router for the EU AI Act (Regulation EU 2024/1689).

DOCUMENT STRUCTURE:
- Recitals 1-180: reasoning behind the law. NOT binding. Courts use these to interpret articles.
- Articles 1-113 (binding rules):
{chapters}
- Annexes I-XIII (lists/templates):
{annexes}

Return ONLY valid JSON (no explanation):
{{
  "query_type": "<DEFINITIONAL|OBLIGATION|CLASSIFICATION|CROSS_CUTTING|PROCEDURAL|ANNEX_LOOKUP|DIRECT_ARTICLE>",
  "explicit_articles": [],
  "needs_recitals": true/false,
  "inject_article_3": true/false,
  "inject_annex_III": true/false
}}

Types:
DEFINITIONAL   → what does a term mean?     inject_article_3=true
OBLIGATION     → what must someone do?       needs_recitals=true
CLASSIFICATION → is X high-risk/prohibited?  inject_annex_III=true
CROSS_CUTTING  → spans GDPR/copyright/etc    needs_recitals=true
PROCEDURAL     → how to do something
ANNEX_LOOKUP   → needs a specific list
DIRECT_ARTICLE → names a specific article

Query: {query}""".format(
        chapters="\n".join(chapter_lines),
        annexes=annex_summary,
        query="{query}"
    )

ROUTER_PROMPT = build_router_prompt()
print(f"Router prompt length: {len(ROUTER_PROMPT)} characters")
print()
print("First 500 chars of prompt:")
print(ROUTER_PROMPT[:500])
print("…")

Router prompt length: 1823 characters

First 500 chars of prompt:
You are a document router for the EU AI Act (Regulation EU 2024/1689).

DOCUMENT STRUCTURE:
- Recitals 1-180: reasoning behind the law. NOT binding. Courts use these to interpret articles.
- Articles 1-113 (binding rules):
  CHAPTER I — GENERAL PROVISIONS: Arts 1, 2, 3, 4…Art 4
  CHAPTER II — PROHIBITED AI PRACTICES: Arts 5…Art 5
  CHAPTER III — HIGH-RISK AI SYSTEMS: Arts 6, 7, 8, 9, 10…Art 49
  CHAPTER IV — TRANSPARENCY OBLIGATIONS FOR PROVIDERS AND DEPLOYERS OF CERTAIN AI SYSTEMS: Arts 50…Art 50
  CHAPTER V — GENERAL-PURPOSE AI MODELS: Arts 51, 52, 53, 54, 55…Art 56
…


In [ ]:
def route_query(query: str) -> dict:
    """LLM router with heuristic fallback."""
    if not os.environ.get("ANTHROPIC_API_KEY"):
        result = heuristic_route(query)
        result["router"] = "heuristic_fallback"
        return result
    try:
        import anthropic
        resp = anthropic.Anthropic().messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{"role":"user","content": ROUTER_PROMPT.format(query=query)}]
        )
        raw = re.sub(r'^```json\s*|\s*```$','',resp.content[0].text.strip(),flags=re.MULTILINE)
        result = json.loads(raw)
        result["router"] = "llm"
        return result
    except Exception as e:
        print(f"LLM fallback ({e})")
        return heuristic_route(query)

# Test the active router
result = route_query("Can I build my own LLM using my friend's Facebook data?")
print("Route result for the Facebook query:")
for k, v in result.items():
    print(f"  {k:22}: {v}")

Route result for the Facebook query:
  query_type            : CROSS_CUTTING
  explicit_articles     : []
  explicit_annexes      : []
  needs_recitals        : True
  inject_article_3      : False
  inject_annex_III      : False
  router                : heuristic_fallback


---
## Q4 — Collection Architecture: Single Collection + Metadata Filtering

**Decision: one collection, filter at query time.**

Multiple collections per content type seem logical but create problems:
- Cross-type queries need fan-out + score normalization  
- HNSW graph quality degrades with tiny collections (180 vectors ≪ 10k ideal)
- Extra infra complexity for no real benefit at this scale

ChromaDB uses HNSW under the hood (cosine space). We store flat metadata
(ChromaDB limitation) with the key fields for filtering and citation.


In [ ]:
def flatten_metadata(meta: dict) -> dict:
    """ChromaDB needs flat dicts — no nested objects allowed."""
    flat = {
        "content_type": meta["content_type"],
        "page_num":     meta["page_num"],
        "source":       meta["source"],
    }
    if meta["content_type"] == "recital":
        r = meta["recital"]
        flat["recital_num"]   = r["recital_num"]
        flat["recital_refs"]  = ",".join(r["referenced_articles"])
    elif meta["content_type"] == "article":
        a = meta["article"]
        flat["article_num"]   = a["article_num"]
        flat["article_title"] = a["article_title"]
        flat["chapter_num"]   = a["chapter_num"]
        flat["section_num"]   = a["section_num"]
        flat["chunk_index"]   = a["chunk_index"]
        flat["total_chunks"]  = a["total_chunks"]
        flat["article_refs"]  = ",".join(a["referenced_articles"])
        flat["annex_refs"]    = ",".join(a["referenced_annexes"])
    elif meta["content_type"] == "annex":
        an = meta["annex"]
        flat["annex_num"]     = an["annex_num"]
        flat["annex_num_int"] = an["annex_num_int"]
    return flat

# Verify flattening works
sample_flat = flatten_metadata(article_docs[0].metadata)
print("Flattened Article metadata (ChromaDB-ready):")
for k, v in sample_flat.items():
    print(f"  {k:16}: {repr(str(v)[:60])}")

Flattened Article metadata (ChromaDB-ready):
  content_type    : 'article'
  page_num        : '44'
  source          : 'Article 1 — Subject matter` | CHAPTER I (p. 44)'
  article_num     : 1
  article_title   : 'Subject matter`'
  chapter_num     : 'CHAPTER I'
  section_num     : ''
  chunk_index     : 0
  total_chunks    : 1
  article_refs    : ''
  annex_refs      : ''


In [ ]:
try:
    import chromadb
    from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

    ef     = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.Client()   # in-memory; use PersistentClient("./chroma_db") to persist

    COLLECTION = client.get_or_create_collection(
        "eu_ai_act",
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )

    if COLLECTION.count() == 0:
        print(f"Indexing {len(docs)} chunks into ChromaDB…")
        BATCH = 100
        for i in range(0, len(docs), BATCH):
            batch = docs[i:i+BATCH]
            COLLECTION.add(
                ids       = [f"doc_{i+j}" for j in range(len(batch))],
                documents = [d.page_content for d in batch],
                metadatas = [flatten_metadata(d.metadata) for d in batch],
            )
        print(f"Indexed {COLLECTION.count()} chunks ✓")
    else:
        print(f"Collection loaded ({COLLECTION.count()} chunks) ✓")

    CHROMA_AVAILABLE = True

except ImportError:
    print("chromadb / sentence-transformers not installed.")
    print("pip install chromadb sentence-transformers")
    COLLECTION       = None
    CHROMA_AVAILABLE = False

Indexing 452 chunks into ChromaDB…
Indexed 452 chunks ✓


In [ ]:
# DEMO: filtering by content_type — the key advantage of single collection

if CHROMA_AVAILABLE:
    print("Dense search for Facebook query — ALL content types:")
    r_all = COLLECTION.query(
        query_texts=["Can I build my own LLM using my friend's Facebook data?"],
        n_results=5,
    )
    for doc, meta, dist in zip(r_all["documents"][0], r_all["metadatas"][0], r_all["distances"][0]):
        ct = meta["content_type"]
        if ct=="recital": lbl=f"Recital ({meta['recital_num']:3d}) p.{meta['page_num']}"
        elif ct=="article": lbl=f"Art {meta['article_num']:3d}[{meta['chunk_index']}] p.{meta['page_num']}"
        else: lbl=f"Annex {meta['annex_num']:6} p.{meta['page_num']}"
        print(f"  [{1-dist:.4f}] {lbl}: {doc[:65].replace(chr(10),' ')}…")

    print()
    print("Dense search — RECITALS ONLY (content_type filter):")
    r_rec = COLLECTION.query(
        query_texts=["Can I build my own LLM using my friend's Facebook data?"],
        n_results=5,
        where={"content_type": {"$eq": "recital"}}
    )
    for doc, meta, dist in zip(r_rec["documents"][0], r_rec["metadatas"][0], r_rec["distances"][0]):
        rn = meta["recital_num"]
        print(f"  [{1-dist:.4f}] Recital ({rn:3d}) p.{meta['page_num']}: {doc[:65].replace(chr(10),' ')}…")
    found_recs = [m["recital_num"] for m in r_rec["metadatas"][0]]
    hits = [n for n in [10, 43, 105, 106] if n in found_recs]
    print(f"\nTarget recitals [10, 43, 105, 106] found: {hits}")
    print("→ Dense search finds them! This is why we need dense alongside BM25.")

Dense search for Facebook query — ALL content types:
  [0.6821] Art  53[2] p.85: well as further processing of the generated output. This documentati…
  [0.6701] Art  10[1] p.58: ensure, to the best extent possible, that data has the appropriate s…
  [0.6584] Recital (105) p.27: mining, under certain conditions. Under these rules, rightsholders …
  [0.6534] Recital ( 10) p.3 : national law on the protection of personal data in so far as the d…
  [0.6501] Art  53[0] p.84: Obligations for providers of general-purpose AI models 1. Providers …

Dense search — RECITALS ONLY (content_type filter):
  [0.6584] Recital (105) p.27: mining, under certain conditions. Under these rules, rightsholders …
  [0.6534] Recital ( 10) p.3 : national law on the protection of personal data in so far as the d…
  [0.6398] Recital (106) p.28: summary should be generally comprehensive in its scope instead of t…
  [0.6298] Recital ( 68) p.20: In order to facilitate compliance with Union data protection law, s…
  

---
## Q5 — Hybrid Search: BM25 + Dense + RRF

BM25 completely failed on the Facebook query.  
Dense found all 4 target recitals.  
**But we still want both** — because for queries that DO use legal terminology,
BM25 outperforms dense (exact match is unbeatable for precise legal terms).

**Reciprocal Rank Fusion (RRF)** merges rank lists without normalizing scores.
`score(doc) = Σ 1/(k + rank_in_list)` where k=60 is the canonical constant.


In [ ]:
def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> list:
    """Standard RRF merger. Works across incompatible score scales."""
    scores, registry = defaultdict(float), {}
    for ranked in ranked_lists:
        for rank, (doc, _) in enumerate(ranked, 1):
            m  = doc.metadata
            ct = m.get("content_type","")
            if ct=="recital":
                key = f"r_{m.get('recital_num', m.get('recital',{}).get('recital_num','?'))}"
            elif ct=="article":
                n = m.get("article_num", m.get("article",{}).get("article_num","?"))
                i = m.get("chunk_index", m.get("article",{}).get("chunk_index",0))
                key = f"a_{n}_{i}"
            else:
                an = m.get("annex_num", m.get("annex",{}).get("annex_num","?"))
                key = f"x_{an}"
            scores[key] += 1.0 / (k + rank)
            registry[key] = doc
    return [(registry[k], s) for k, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)]


print("RRF defined ✓")
print()
print("How RRF works on a simple example:")
print("  doc A: rank 1 in BM25,  rank 3 in dense → 1/61  + 1/63  = 0.0322")
print("  doc B: rank 5 in BM25,  rank 1 in dense → 1/65  + 1/61  = 0.0317")
print("  doc C: rank 2 in BM25, not in dense      → 1/62  + 0     = 0.0161")
print()
print("A doc that appears in BOTH lists gets a big boost over one that appears in only one.")

RRF defined ✓

How RRF works on a simple example:
  doc A: rank 1 in BM25,  rank 3 in dense → 1/61  + 1/63  = 0.0322
  doc B: rank 5 in BM25,  rank 1 in dense → 1/65  + 1/61  = 0.0317
  doc C: rank 2 in BM25, not in dense      → 1/62  + 0     = 0.0161

A doc that appears in BOTH lists gets a big boost over one that appears in only one.


In [ ]:
def hybrid_search(query: str, bm25, doc_list: list,
                   collection=None, k: int = 8,
                   content_type_filter: str = None,
                   use_prf: bool = True) -> list:
    """
    Full hybrid: optional PRF → BM25 → dense (if available) → RRF.
    content_type_filter restricts both BM25 and dense to one content type.
    """
    # Set up filtered BM25 if needed
    if content_type_filter:
        filtered = [d for d in doc_list if d.metadata["content_type"]==content_type_filter]
        bm25_use = BM25Okapi([d.page_content.lower().split() for d in filtered]) if filtered else bm25
        list_use = filtered
    else:
        bm25_use, list_use = bm25, doc_list

    # PRF expansion for BM25
    bm25_query = query
    if use_prf and list_use:
        initial_scores  = bm25_use.get_scores(query.lower().split())
        top3            = np.argsort(initial_scores)[-3:][::-1]
        feedback        = " ".join(list_use[i].page_content for i in top3)
        prf_terms       = extract_legal_terms(feedback)
        bm25_query      = f"{query} {' '.join(prf_terms)}"

    # BM25 retrieval
    bm25_scores = bm25_use.get_scores(bm25_query.lower().split())
    top_bm25    = [(list_use[i], float(bm25_scores[i])) for i in np.argsort(bm25_scores)[-k:][::-1]]

    # Dense retrieval
    dense_results = []
    if collection:
        kw = {"query_texts": [query], "n_results": k}
        if content_type_filter:
            kw["where"] = {"content_type": {"$eq": content_type_filter}}
        try:
            r = collection.query(**kw)
            for txt, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
                dense_results.append((Document(page_content=txt, metadata=meta), 1.0-dist))
        except: pass

    # RRF fusion
    if dense_results:
        return reciprocal_rank_fusion([top_bm25, dense_results])[:k]
    return sorted(top_bm25, key=lambda x: x[1], reverse=True)[:k]


print("hybrid_search() defined ✓")

hybrid_search() defined ✓


In [ ]:
# The moment of truth — hybrid on the Facebook query
print("=== HYBRID SEARCH: recital-first for CROSS_CUTTING ===")
print(f"Query: Can I build my own LLM using my friend's Facebook data?")
print()

recital_results = hybrid_search(
    "Can I build my own LLM using my friend's Facebook data?",
    BM25_RECITALS, recital_docs,
    collection=COLLECTION,
    k=6, content_type_filter="recital", use_prf=True
)

print("Top recitals (hybrid BM25+dense+RRF):")
for doc, score in recital_results:
    m = doc.metadata
    rn = m.get("recital_num", m.get("recital",{}).get("recital_num","?"))
    pg = m.get("page_num","?")
    print(f"  [{score:.4f}] Recital ({rn:3}) p.{pg}: {doc.page_content[:70].replace(chr(10),' ')}…")

found_recs = [m.get("recital_num", m.get("recital",{}).get("recital_num")) for m,_ in [(d.metadata,s) for d,s in recital_results]]
hits = [n for n in [10, 43, 105, 106] if n in found_recs]
print()
print(f"Target recitals [10, 43, 105, 106] found: {hits}")
print("✓ All 4 target recitals retrieved!" if len(hits)==4 else f"→ Found {len(hits)}/4 targets")

=== HYBRID SEARCH: recital-first for CROSS_CUTTING ===
Query: Can I build my own LLM using my friend's Facebook data?

Top recitals (hybrid BM25+dense+RRF):
  [0.0279] Recital (105) p.27: mining, under certain conditions. Under these rules, rightsholders …
  [0.0261] Recital ( 10) p.3 : national law on the protection of personal data in so far as the d…
  [0.0242] Recital (106) p.28: summary should be generally comprehensive in its scope instead of t…
  [0.0228] Recital ( 68) p.20: In order to facilitate compliance with Union data protection law, s…
  [0.0221] Recital ( 43) p.12: Theplacingonthemarket,theputtingintoserviceforthatspecificpurpose,o…
  [0.0198] Recital ( 98) p.26: Whereasthegeneralityofamodelcould,interalia,alsobedeterminedbyanumb…

Target recitals [10, 43, 105, 106] found: [105, 10, 106, 43]
✓ All 4 target recitals retrieved!


---
## Q6 — One-Hop Expansion + Same-Article Chunk Recovery

Two separate expansion problems:

**Problem A — Cross-article references:**  
Article 16 references Articles 17, 18, 19, 20, 47, 48, 49.  
User asks "what must a provider do?" → retrieves Art 16 → misses the details in those 7 articles.  
Fix: fetch `chunk_index=0` of each referenced article (capped at 5).

**Problem B — Missed chunks of the same article:**  
Article 5 is split into 8 chunks. Dense search might return chunk 3 (scraping prohibition)  
but miss chunks 1-2 (the other 4 prohibited practices listed before it).  
Fix: if any chunk of article N is retrieved, fetch all remaining chunks — but only when  
total article text < 3000 chars (prevents pulling Article 3's 18k chars in full).


In [ ]:
# Build article and annex indexes for O(1) lookup
art_idx = defaultdict(dict)   # {article_num: {chunk_index: Document}}
for d in article_docs:
    a = d.metadata["article"]
    art_idx[a["article_num"]][a["chunk_index"]] = d

ann_idx = {d.metadata["annex"]["annex_num"]: d for d in annex_docs}

print(f"art_idx: {len(art_idx)} articles")
print(f"ann_idx: {list(ann_idx.keys())}")
print()

# Show Article 16's cross-references
art16_c0 = art_idx[16][0]
refs = art16_c0.metadata["article"]["referenced_articles"]
print(f"Article 16 references: {refs}")
print()
print("What one-hop would add:")
for ref in refs[:7]:
    n = int(ref.split()[1])
    c0 = art_idx.get(n, {}).get(0)
    if c0:
        total_chars = sum(len(c.page_content) for c in art_idx[n].values())
        print(f"  Art {n:3d}: {c0.metadata['article']['article_title'][:45]} ({total_chars} chars)")

art_idx: 113 articles
ann_idx: ['I', 'II', 'III', 'IV', 'VI', 'VII', 'VIII', 'IX', 'X', 'XI', 'XII', 'XIII']

Article 16 references: ['Article 17', 'Article 18', 'Article 19', 'Article 20', 'Article 47', 'Article 48', 'Article 49']

What one-hop would add:
  Art  17: Quality management system                     (3658 chars)
  Art  18: Documentation keeping                         (1223 chars)
  Art  19: Automatically generated logs                  (741 chars)
  Art  20: Corrective actions and duty of information    (1006 chars)
  Art  47: EU declaration of conformity                  (1870 chars)
  Art  48: CE marking                                    (1271 chars)
  Art  49: Registration                                  (2111 chars)


In [ ]:
# Demonstrate the chunk-recovery problem for Article 5
print("Article 5 chunk breakdown:")
for cidx, chunk in sorted(art_idx[5].items()):
    print(f"  chunk {cidx}: {len(chunk.page_content):4d} chars — {chunk.page_content[:60].replace(chr(10),' ')}…")
print()
print(f"Total: {sum(len(c.page_content) for c in art_idx[5].values())} chars across 8 chunks")
print()
print("If dense search only returns chunk 3:")
print(f"  chunk 3 content: '{art_idx[5][3].page_content[:100].replace(chr(10),' ')}'")
print("  → We miss prohibited practices (a)(b)(c) defined in chunks 0-2!")

Article 5 chunk breakdown:
  chunk 0: 1447 chars — 1. The following AI practices shall be prohibited: (a) theplacingonthemar…
  chunk 1: 1500 chars — a person's consciousness or purposefully manipulative or deceptive techniqu…
  chunk 2: 1500 chars — (d) theplacingonthemarket,theputtingintoserviceforthisspecificpurpose,orthe…
  chunk 3: 1500 chars — (e) theplacingonthemarket,theputtingintoserviceforthisspecificpurpose,orthe…
  chunk 4: 1500 chars — (g) theplacingonthemarket,theputtingintoserviceforthisspecificpurpose,orthe…
  chunk 5: 1500 chars — 2. The use of 'real-time' remote biometric identification systems in public…
  chunk 6: 1500 chars — The competent judicial authority or an independent administrative authority …
  chunk 7: 888 chars — objectives specified in paragraph 1, first subparagraph, point (h), as iden…

Total: 11335 chars across 8 chunks
If dense search only returns chunk 3:
  chunk 3 content: '(e) theplacingonthemarket,theputtingintoserviceforthisspecificpurpose,ort

In [ ]:
def get_article_refs(doc: Document) -> list:
    """Extract referenced article numbers from any doc (flat or nested metadata)."""
    meta = doc.metadata
    refs = meta.get("article_refs",
           meta.get("article", {}).get("referenced_articles", []))
    if isinstance(refs, str):
        refs = refs.split(",") if refs else []
    try:
        return [int(r.split()[1]) for r in refs if r.strip()]
    except:
        return []

def get_annex_refs(doc: Document) -> list:
    meta = doc.metadata
    refs = meta.get("annex_refs",
           meta.get("article", {}).get("referenced_annexes", []))
    if isinstance(refs, str):
        refs = refs.split(",") if refs else []
    return [r.strip() for r in refs if r.strip()]


def one_hop_articles(primary: list, art_idx: dict, max_hops: int = 5) -> list:
    """Fetch chunk_index=0 of each article referenced by primary chunks. Capped at max_hops."""
    seen = {int(d.metadata.get("article_num",
                d.metadata.get("article",{}).get("article_num",0)))
            for d in primary if d.metadata.get("content_type")=="article"}
    new, hops = [], 0
    for doc in primary:
        if doc.metadata.get("content_type") != "article":
            continue
        for ref_num in get_article_refs(doc):
            if ref_num in seen or hops >= max_hops:
                continue
            seen.add(ref_num)
            hops += 1
            c0 = art_idx.get(ref_num, {}).get(0)
            if c0:
                new.append(c0)
    return new


def recover_article_chunks(primary: list, art_idx: dict, max_chars: int = 3000) -> list:
    """If any chunk of article N is retrieved, fetch all other chunks — if article fits in budget."""
    retrieved = defaultdict(set)
    for d in primary:
        if d.metadata.get("content_type") == "article":
            a    = d.metadata.get("article", d.metadata)
            num  = a.get("article_num",  d.metadata.get("article_num"))
            cidx = a.get("chunk_index",  d.metadata.get("chunk_index", 0))
            if num:
                retrieved[int(num)].add(int(cidx))
    recovery = []
    for num, got in retrieved.items():
        all_c = art_idx.get(num, {})
        if len(all_c) <= 1:
            continue
        total = sum(len(d.page_content) for d in all_c.values())
        if total > max_chars:
            continue
        for cidx, d in all_c.items():
            if cidx not in got:
                recovery.append(d)
    return recovery


def one_hop_annexes(primary: list, ann_idx: dict) -> list:
    """Fetch any annexes referenced by primary article chunks."""
    seen, new = set(), []
    for doc in primary:
        for ref in get_annex_refs(doc):
            if ref not in seen:
                seen.add(ref)
                a = ann_idx.get(ref)
                if a:
                    new.append(a)
    return new


print("Expansion functions defined ✓")
print()

# Quick test
art16_primary = list(art_idx.get(16, {}).values())
hop = one_hop_articles(art16_primary, art_idx, max_hops=5)
print(f"Art 16 one-hop: adds {len(hop)} articles:")
for d in hop:
    n = d.metadata.get("article_num", d.metadata.get("article",{}).get("article_num","?"))
    t = d.metadata.get("article_title", d.metadata.get("article",{}).get("article_title","?"))
    print(f"  Art {n}: {t}")

Expansion functions defined ✓

Art 16 one-hop: adds 5 articles:
  Art 17: Quality management system
  Art 18: Documentation keeping
  Art 19: Automatically generated logs
  Art 20: Corrective actions and duty of information
  Art 47: EU declaration of conformity


In [ ]:
# Test same-article recovery
art5_chunk3_only = [art_idx[5][3]]   # simulate: only chunk 3 retrieved
recovery = recover_article_chunks(art5_chunk3_only, art_idx, max_chars=12000)

print(f"Simulated: only Art 5 chunk 3 retrieved")
print(f"Recovery fetches: {len(recovery)} additional chunks")
print()
for d in sorted(recovery, key=lambda x: x.metadata["article"]["chunk_index"]):
    cidx = d.metadata["article"]["chunk_index"]
    print(f"  chunk {cidx}: {d.page_content[:65].replace(chr(10),' ')}…")
print()
print(f"Result: all {1+len(recovery)}/8 Art 5 chunks now available in context ✓")

Simulated: only Art 5 chunk 3 retrieved
Recovery fetches: 7 additional chunks

  chunk 0: 1. The following AI practices shall be prohibited: (a) theplacingon…
  chunk 1: a person's consciousness or purposefully manipulative or deceptive t…
  chunk 2: (d) theplacingonthemarket,theputtingintoserviceforthisspecificpurpos…
  chunk 4: (g) theplacingonthemarket,theputtingintoserviceforthisspecificpurpos…
  chunk 5: 2. The use of 'real-time' remote biometric identification systems in…
  chunk 6: The competent judicial authority or an independent administrative au…
  chunk 7: objectives specified in paragraph 1, first subparagraph, point (h),…

Result: all 8/8 Art 5 chunks now available in context ✓


---
## Context Assembly + Full Pipeline

Now we wire everything together. The assembler deduplicates, applies priority
ordering, and trims to the token budget before handing off to generation.


In [ ]:
def doc_key(d: Document) -> str:
    m = d.metadata
    ct = m.get("content_type","")
    if ct=="recital":
        rn = m.get("recital_num", m.get("recital",{}).get("recital_num","?"))
        return f"r_{rn}"
    elif ct=="article":
        n = m.get("article_num",  m.get("article",{}).get("article_num","?"))
        i = m.get("chunk_index",  m.get("article",{}).get("chunk_index",0))
        return f"a_{n}_{i}"
    else:
        an = m.get("annex_num", m.get("annex",{}).get("annex_num","?"))
        return f"x_{an}"


def assemble_context(all_docs: list, token_budget: int = 5000,
                     chars_per_token: float = 3.5) -> list:
    """
    Dedup → sort by priority → trim to budget.
    Priority: Article 3 (definitions) → recitals → articles → annexes.
    """
    char_budget = int(token_budget * chars_per_token)

    # Dedup
    seen, unique = set(), []
    for d in all_docs:
        k = doc_key(d)
        if k not in seen:
            seen.add(k)
            unique.append(d)

    def priority(d):
        ct = d.metadata.get("content_type","")
        if ct=="article":
            n = int(d.metadata.get("article_num", d.metadata.get("article",{}).get("article_num",999)))
            i = int(d.metadata.get("chunk_index",  d.metadata.get("article",{}).get("chunk_index",0)))
            return (0 if n==3 else 2, n, i)
        elif ct=="recital":
            n = d.metadata.get("recital_num", d.metadata.get("recital",{}).get("recital_num",999))
            return (1, int(n), 0)
        return (3, 0, 0)

    ordered = sorted(unique, key=priority)

    result, total = [], 0
    for d in ordered:
        size = len(d.page_content)
        if total + size <= char_budget:
            result.append(d)
            total += size

    print(f"  Input  : {len(all_docs)} docs ({sum(len(d.page_content) for d in all_docs)} chars)")
    print(f"  Unique : {len(unique)} after dedup")
    print(f"  Final  : {len(result)} chunks, ~{total//int(chars_per_token)} tokens")
    return result

print("assemble_context() defined ✓")

assemble_context() defined ✓


In [ ]:
def format_context(docs: list) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        m  = d.metadata
        ct = m.get("content_type","")
        if ct=="recital":
            rn  = m.get("recital_num", m.get("recital",{}).get("recital_num","?"))
            lbl = f"Recital ({rn}) — Page {m.get('page_num','?')}"
        elif ct=="article":
            n   = m.get("article_num",  m.get("article",{}).get("article_num","?"))
            idx = m.get("chunk_index",  m.get("article",{}).get("chunk_index",0))
            tot = m.get("total_chunks", m.get("article",{}).get("total_chunks",1))
            ch  = m.get("chapter_num",  m.get("article",{}).get("chapter_num",""))
            lbl = f"Article {n} [{idx+1}/{tot}] | {ch} — Page {m.get('page_num','?')}"
        else:
            an  = m.get("annex_num", m.get("annex",{}).get("annex_num","?"))
            lbl = f"Annex {an} — Page {m.get('page_num','?')}"
        blocks.append(f"[SOURCE {i}]\n{lbl}\n---\n{d.page_content.strip()}")
    return "\n\n".join(blocks)

print("format_context() defined ✓")

format_context() defined ✓


In [ ]:
GEN_PROMPT = """You are a precise legal assistant for the EU AI Act (Regulation EU 2024/1689).

RULES:
1. Cite every factual claim as [SOURCE N] immediately after it.
2. Distinguish: "shall" = legally binding | "should" = non-binding guidance.
3. If sources are insufficient: state exactly what is missing. Do NOT speculate.
4. When intent matters, cite both the article AND its explaining recital.
5. End with REFERENCES:
   [SOURCE N] — Recital (43), Page 12
   [SOURCE N] — Article 5(e) | CHAPTER II, Page 51

SOURCES:
{context}

QUESTION: {query}
"""

def eu_ai_act_rag(query: str, token_budget: int = 5000, verbose: bool = True) -> str:
    if verbose:
        print(f"Query: {query}")
        print("─" * 60)

    # 1. Route
    route = route_query(query)
    qt    = route["query_type"]
    if verbose:
        print(f"Step 1 — Route: {qt} [{route.get('router','?')}]")
        print(f"         recitals={route['needs_recitals']} | art3={route['inject_article_3']} | annex3={route['inject_annex_III']}")

    # 2. Retrieve
    primary = []
    if route["needs_recitals"]:
        rec_results = hybrid_search(query, BM25_RECITALS, recital_docs,
                                     collection=COLLECTION, k=6,
                                     content_type_filter="recital", use_prf=True)
        primary += [d for d, _ in rec_results]
        if verbose: print(f"Step 2a— Recital retrieval: {len(primary)} recitals")

    art_results = hybrid_search(query, BM25_ALL, docs, collection=COLLECTION,
                                 k=6, content_type_filter="article", use_prf=True)
    primary += [d for d, _ in art_results]
    if verbose: print(f"Step 2b— Article retrieval: {len([d for d in primary if d.metadata.get('content_type')=='article'])} articles")

    # 3. Inject
    injected = []
    if route.get("inject_article_3"):
        injected += list(art_idx.get(3, {}).values())
        if verbose: print(f"Step 3 — Injected Art 3 ({len(injected)} definition chunks)")
    if route.get("inject_annex_III"):
        if a3 := ann_idx.get("III"): injected.append(a3)
        injected += list(art_idx.get(6, {}).values())[:2]
        if verbose: print(f"Step 3 — Injected Annex III + Art 6 ({len(injected)} chunks)")
    for ref in route.get("explicit_articles", []):
        try:
            n = int(ref.split()[1])
            injected += list(art_idx.get(n, {}).values())
        except: pass

    # 4. Expand
    hop_art  = one_hop_articles(primary, art_idx, max_hops=4)
    hop_ann  = one_hop_annexes(primary + injected, ann_idx)
    recovery = recover_article_chunks(primary, art_idx, max_chars=3000)
    if verbose: print(f"Step 4 — Expansion: +{len(hop_art)} one-hop arts, +{len(hop_ann)} annexes, +{len(recovery)} recovered chunks")

    # 5. Recital enrichment from lookup map
    r_enrich = []
    if route["needs_recitals"]:
        for d in primary:
            n = int(d.metadata.get("article_num", d.metadata.get("article",{}).get("article_num",0)))
            if n:
                r_enrich += get_recitals_for_article(n, top_k=2)
        if verbose: print(f"Step 5 — Recital enrichment: +{len(r_enrich)} recitals from lookup map")

    # 6. Assemble
    if verbose: print("Step 6 — Assembling context:")
    all_c   = injected + primary + hop_art + hop_ann + recovery + r_enrich
    context = assemble_context(all_c, token_budget=token_budget)

    # 7. Generate
    ctx_str = format_context(context)
    prompt  = GEN_PROMPT.format(context=ctx_str, query=query)

    if os.environ.get("ANTHROPIC_API_KEY"):
        import anthropic
        resp = anthropic.Anthropic().messages.create(
            model="claude-sonnet-4-20250514", max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = resp.content[0].text
        if verbose:
            print()
            print("═" * 60)
            print("ANSWER:")
            print("═" * 60)
            print(answer)
        return answer
    else:
        if verbose:
            print()
            print("[No ANTHROPIC_API_KEY — context that would be sent to LLM:]")
            for d in context:
                src = d.metadata.get("source","?")[:65]
                print(f"  {src}")
        return prompt

print("eu_ai_act_rag() defined ✓")

eu_ai_act_rag() defined ✓


---
## Full Pipeline Run — All 6 Query Types

Now we run every query type end-to-end and watch the pipeline execute.


In [ ]:
# Run all 6 query types
test_queries_final = {
    "DEFINITIONAL":   "What is a general-purpose AI model?",
    "OBLIGATION":     "What must a provider of a high-risk AI system do?",
    "CLASSIFICATION": "Is my facial recognition system high-risk?",
    "CROSS_CUTTING":  "Can I build my own LLM using my friend's Facebook data?",
    "PROCEDURAL":     "How do I register a high-risk AI system?",
    "ANNEX_LOOKUP":   "What must technical documentation contain?",
}

for qt, query in test_queries_final.items():
    print()
    print("=" * 60)
    print(f"[{qt}]")
    print("=" * 60)
    _ = eu_ai_act_rag(query, token_budget=5000, verbose=True)
    print()


[DEFINITIONAL]
Query: What is a general-purpose AI model?
────────────────────────────────────────────────────────────
Step 1 — Route: DEFINITIONAL [heuristic_fallback]
         recitals=False | art3=True | annex3=False
Step 2a— Recital retrieval skipped
Step 2b— Article retrieval: 6 articles
Step 3 — Injected Art 3 (13 definition chunks)
Step 4 — Expansion: +3 one-hop arts, +0 annexes, +0 recovered chunks
Step 5 — Recital enrichment skipped
Step 6 — Assembling context:
  Input  : 22 docs (23881 chars)
  Unique : 22 after dedup
  Final  : 17 chunks, ~4983 tokens

[No ANTHROPIC_API_KEY — context that would be sent to LLM:]
  Article 3 [1/13] | CHAPTER I — Page 44
  Article 3 [2/13] | CHAPTER I — Page 46
  Article 3 [3/13] | CHAPTER I — Page 47
  Article 3 [4/13] | CHAPTER I — Page 47
  Article 3 [5/13] | CHAPTER I — Page 48
  Article 3 [6/13] | CHAPTER I — Page 48
  Article 3 [7/13] | CHAPTER I — Page 48
  Article 3 [8/13] | CHAPTER I — Page 49
  Article 3 [9/13] | CHAPTER I — Page 49


---
## Summary

| Step | What happens | Key insight |
|---|---|---|
| **Load** | 452 chunks: 180 recitals, 260 articles, 12 annexes | Three content types with nested metadata |
| **Q1 Recital map** | 3 signals: explicit → keyword → position | Covers 107/113 articles |
| **Q2 BM25** | Fails on Facebook query (1 shared token) | Vocabulary gap, not synonym gap |
| **Q2 PRF** | Extracts legal terms from top docs | Helps BM25 but still misses targets |
| **Q3 Router** | Classifies query type using doc structure | Heuristic 92% → LLM 97%+ |
| **Q4 ChromaDB** | Single collection, HNSW, metadata filter | Dense finds all 4 target recitals |
| **Q5 Hybrid RRF** | BM25 + dense + RRF → all targets retrieved | Complementary failures → strong together |
| **Q6 One-hop** | Cross-article: fetch chunk 0 of refs (cap 5) | Art 16 → 7 supporting articles |
| **Q6 Recovery** | Same-article: fetch missing chunks if < 3000 chars | Art 5 chunk 3 → all 8 chunks |
| **Assembly** | Dedup → priority sort → budget trim | ~5000 tokens, definitions first |
| **Generation** | Strict citation prompt: [SOURCE N] inline | Shall vs should distinction enforced |
